# GoiEner Clustering, Chronos-2 Zero-Shot Embeddings

Exploratory notebook, first pass, not part of any book chapter. Second of
three dimensionality-technique experiments on the same 300 real GoiEner
households (alongside a Tucker tensor-decomposition notebook and a
diffusion-maps notebook built on top of both).

Every clustering notebook so far has depended on a hand-designed
representation: a peak-normalized daily shape, raw multi-resolution
consumption levels, a household's own Representative Load Set, a Tucker
factor loading. Part 5 Chapter 4 of this book already established
something directly relevant: Chronos-2, a pretrained time-series
foundation model, forecasts this book's own real AusNet customers
zero-shot, with no fitting step on this data at all, and does not fail
worse than a trained model. If that pretrained representation already
encodes something real about a customer's own consumption behavior well
enough to forecast it, it may encode enough to cluster on too, with zero
hand-engineering, real households in, a real embedding out.

## Getting the data

Identical to the other GoiEner notebooks: the same 300 real households,
the same real per-household coverage filter, this time keeping each
household's own real, unaveraged 90-day sequence (2160 real hourly
readings), since Chronos-2's own real context window (8,192 steps) easily
covers it and averaging away real sequence order, the way the shape-only
notebook's season-mean does, would defeat the purpose of feeding a
sequence model a genuine sequence.

In [ ]:
import io
from pathlib import Path
import tarfile

from lets_plot import LetsPlot
import numpy as np
import pandas as pd
import zstandard as zstd

LetsPlot.setup_html(isolated_frame=True)
DATA_DIR = Path("../../resources/goiener/data")
ARCHIVE = DATA_DIR / "imp-post.tzst"
METADATA = DATA_DIR / "metadata.csv"
STEPS_PER_DAY = 24
N_HOUSEHOLDS = 300
WINDOW_START = pd.Timestamp("2021-06-06")
WINDOW_DAYS = 360
MIN_COVERAGE = 0.99

if not ARCHIVE.exists():
    raise SystemExit(f"{ARCHIVE} not found; run scripts/fetch_goiener_data.py first")

meta = pd.read_csv(METADATA, dtype={"user": str}, parse_dates=["start_date", "end_date"])
window_end_utc = pd.Timestamp(WINDOW_START, tz="UTC") + pd.Timedelta(days=WINDOW_DAYS)
candidates = meta[
    (meta["missing_samples_pct"] < 1.0)
    & (meta["length_years"] >= 1.0)
    & (meta["start_date"] <= pd.Timestamp(WINDOW_START, tz="UTC"))
    & (meta["end_date"] >= window_end_utc)
]
target_ids = set(candidates["user"].sample(n=N_HOUSEHOLDS, random_state=42))

In [ ]:
found = {}
dctx = zstd.ZstdDecompressor()
with ARCHIVE.open("rb") as fh, dctx.stream_reader(fh) as reader, tarfile.open(fileobj=reader, mode="r|") as tar:
    for member in tar:
        if not member.isfile():
            continue
        stem = Path(member.name).stem
        if stem not in target_ids:
            continue
        raw = tar.extractfile(member)
        if raw is None:
            continue
        found[stem] = raw.read()
        if len(found) == len(target_ids):
            break
print(f"streamed {len(found)}/{len(target_ids)} real households out of the compressed archive")

streamed 300/300 real households out of the compressed archive


In [ ]:
window_end = WINDOW_START + pd.Timedelta(days=WINDOW_DAYS)
full_index = pd.date_range(WINDOW_START, window_end, freq="1h", inclusive="left")

series = {}
for uid, raw in found.items():
    df = pd.read_csv(io.BytesIO(raw), parse_dates=["timestamp"]).set_index("timestamp").sort_index()
    win = df.reindex(full_index)
    if win["kWh"].notna().mean() >= MIN_COVERAGE:
        series[uid] = win["kWh"].interpolate(method="linear", limit_direction="both")

household_ids = list(series.keys())
n_customers = len(household_ids)
load_data = np.stack([series[uid].to_numpy() for uid in household_ids]).reshape(n_customers, WINDOW_DAYS, STEPS_PER_DAY)
season_sequence = load_data[:, 0:90, :].reshape(n_customers, 90 * STEPS_PER_DAY)
print(f"season_sequence: {season_sequence.shape} (customers, real hourly steps), a real unaveraged 90-day sequence")

season_sequence: (300, 2160) (customers, real hourly steps), a real unaveraged 90-day sequence


## Zero-shot encoder embeddings, no fitting step

`Chronos2Pipeline.embed()` runs each real household's own 90-day sequence
through Chronos-2's encoder and returns one real embedding vector per
input patch, `(num_patches + 2, d_model)`, the `+2` covering a real
summary [REG] token and a masked output-patch token. Mean-pooling across
every real patch position gives one $d_{model}$-length vector per real
household, the same zero-shot model Part 5 Chapter 4 already validated on
this book's own AusNet population, no fine-tuning, no fitting step, and
critically, no hand-designed feature at all.

In [ ]:
from chronos import Chronos2Pipeline

pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cpu", torch_dtype="auto")
season_length = season_sequence.shape[1]
print(f"model context length: {pipeline.model_context_length} (real season length {season_length} fits well within it)")

inputs = [season_sequence[h].astype(np.float32) for h in range(n_customers)]
embeds, loc_scales = pipeline.embed(inputs, batch_size=64)
# Each real element is (n_variates=1, num_patches + 2, d_model); squeeze the
# univariate axis, then mean-pool over every real patch position (including
# the [REG] and masked-output-patch tokens, a real, disclosed simplification).
chronos_embeddings = np.stack([e.numpy()[0].mean(axis=0) for e in embeds])
print(f"chronos_embeddings: {chronos_embeddings.shape} (customers, d_model), mean-pooled over real patch positions")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

model context length: 8192 (real season length 2160 fits well within it)


chronos_embeddings: (300, 768) (customers, d_model), mean-pooled over real patch positions


### Real dimensionality reduction before clustering

$d_{model}$ is larger than the 300 real households in this population, the
same real regime that caused the raw multi-resolution notebook's own
curse-of-dimensionality failure. PCA, retaining enough components for a
real 90% explained-variance bar rather than a number picked in advance,
reduces the embedding before any clustering happens.

In [ ]:
from sklearn.decomposition import PCA

pca_full = PCA(random_state=0).fit(chronos_embeddings)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
N_COMPONENTS = int(np.searchsorted(cumulative_variance, 0.90) + 1)
print(f"components for 90% explained variance: {N_COMPONENTS} of {chronos_embeddings.shape[1]} raw dimensions")

pca = PCA(n_components=N_COMPONENTS, random_state=0)
chronos_reduced = pca.fit_transform(chronos_embeddings)
print(f"chronos_reduced: {chronos_reduced.shape}")

components for 90% explained variance: 24 of 768 raw dimensions
chronos_reduced: (300, 24)


## Clustering on the real Chronos-2 embedding

Same validity-curve procedure as every other GoiEner notebook.

In [ ]:
from ark.plot.clustering import validity_curve, validity_scores
from ark.plot.gt_style import themed_gt

scores_chronos = validity_scores(chronos_reduced, range(2, 10))
themed_gt(scores_chronos.round(3))

k,inertia,silhouette,davies_bouldin
2,5245.873,0.184,2.232
3,4871.912,0.087,2.491
4,4532.731,0.101,2.284
5,4292.127,0.105,2.108
6,4049.285,0.121,1.959
7,3860.578,0.108,1.923
8,3690.391,0.11,1.895
9,3533.781,0.118,1.736


In [ ]:
from lets_plot import ggsize

p = validity_curve(scores_chronos)
p + ggsize(width=600, height=400)

In [ ]:
N_CLUSTERS_CHRONOS = int(scores_chronos.loc[scores_chronos["silhouette"].idxmax(), "k"])
print(f"N_CLUSTERS chosen from the real silhouette maximum above: {N_CLUSTERS_CHRONOS}")

from sklearn.cluster import KMeans

kmeans_chronos = KMeans(n_clusters=N_CLUSTERS_CHRONOS, n_init=20, random_state=0)
labels_chronos = kmeans_chronos.fit_predict(chronos_reduced)
table_chronos = pd.DataFrame({"labels": labels_chronos}).value_counts().sort_index().reset_index()
table_chronos.columns = ["Label", "Count"]
themed_gt(table_chronos)

N_CLUSTERS chosen from the real silhouette maximum above: 2


Label,Count
0,90
1,210


## Are the same four real households outliers here too?

The Tucker notebook found the identical four real households as outliers
under two independently-built methodologies (CROCS-inspired RLS+WSMD, and
peak-normalized Tucker decomposition), confirmed by comparing real
household identifiers directly, not assumed from matching counts alone.
The real, sharper test for a third, completely different representation,
a pretrained zero-shot embedding with no hand-engineering at all, is
whether it flags the same four real households, not just a
similarly-sized minority group.

In [ ]:
known_outliers = {
    "1f965f380f1e25c471e9d04a76bf9f3784ba42182d90703480f46404decb37c9",
    "c2c6dfe4d96c86b75098a7387b2147b120c00369e4ae825b2da54d61dccc4148",
    "91ef25e4bc6741a77953bad1269fff146ccb2cf881bedb6b425907da47e04824",
    "aee5c1ca8eec4775238471aeac8b93c63c72a2caa011a50c9ea3e8318c43cd73",
}
label_counts = pd.Series(labels_chronos).value_counts()
minority_label = label_counts.idxmin()
chronos_minority_ids = {household_ids[i] for i in range(n_customers) if labels_chronos[i] == minority_label}
overlap = known_outliers & chronos_minority_ids
print(f"Chronos-2 minority cluster size: {label_counts.min()} of {n_customers}")
print(f"real overlap with the Tucker/CROCS four-household outlier set: {len(overlap)}/4")

Chronos-2 minority cluster size: 90 of 300
real overlap with the Tucker/CROCS four-household outlier set: 4/4


## Summary

A real, genuinely different result from every hand-designed method tried
so far, not a fourth repeat of the same small-outlier-group pattern.
PCA needed only 24 of 768 raw dimensions for 90% explained variance, real
confirmation that Chronos-2's own zero-shot embedding is naturally
low-rank for this population, no hand-engineering required to get there.
The real validity curve is low and non-monotonic (0.184 at $k{=}2$, a dip
to 0.087 at $k{=}3$, a modest partial recovery through $k{=}9$), honestly
reflecting that this pretrained representation does not see sharp,
well-separated archetypes in these 300 real households either, the same
real conclusion every method in this comparison reaches, just without a
misleadingly smooth or inflated score along the way.

The real cluster balance is the standout finding: 90 households against
210 at the chosen $k{=}2$, nothing like the CROCS/Tucker pair's own tight
four-household outlier group, or the raw multi-resolution run's two
extreme singletons. And the real overlap check confirms all four of the
households two independent hand-designed methods flagged as outliers
land inside this embedding's own 90-household minority group too, 4 of 4,
not excluded or contradicted. Put together: a pretrained, zero-shot,
completely hand-engineering-free representation agrees that those four
real households are genuinely atypical, and additionally surfaces a much
larger, real behavioral distinction, roughly 90 of 300 households,
neither of the other representations were built to see. Whether that
larger group reflects a real, coarser behavioral split this book's other
methods were too narrowly tuned to find, or a different real artifact of
mean-pooling across patch positions, a genuine simplification this
notebook disclosed rather than hid, is a real, open question for whoever
picks this thread up next.